# Análisis de la red ibérica en PyPSA-Eur

Objetivos:

- Cargar la red `solved_2050.nc`.
- Seleccionar España y Portugal.
- Mantener las interconexiones con Francia.
- No modificar el archivo original.
- Visualizar la red filtrada.
- Dejar preparadas las tablas para analizar buses y líneas.

In [4]:
import pypsa
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

Cargo la red de España, Portugal y Francia

In [6]:
ruta_solved_2050 = r"C:\workspace\EAGLE\external\pypsa-eur\resources\eagle-iberia\networks\solved_2050.nc"

n_solved_2050 = pypsa.Network(ruta_solved_2050)

n_solved_2050

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\workspace\\EAGLE\\external\\pypsa-eur\\resources\\eagle-iberia\\networks\\solved_2050.nc'

In [ ]:
n_solved_2050.buses["country"].value_counts()

Seleccion de buses y lineas de España + Portugal y conservar solamente las conexiones hacia Francia.

In [ ]:
# Seleccionamos únicamente los buses que pertenecen a España (ES) o Portugal (PT).
# n_solved_2050.buses es una tabla (DataFrame) donde cada fila representa un bus.
# La columna "country" indica a qué país pertenece cada uno.

buses_iberia = n_solved_2050.buses[
    n_solved_2050.buses.country.isin(["ES", "PT"])
]

# Mostramos la tabla resultante con todos los buses de España y Portugal.
buses_iberia


# Guardamos los identificadores de esos buses.
# El índice de n_solved_2050.buses contiene el nombre/ID único de cada bus.
# Nos servirá para saber qué líneas tienen sus extremos dentro de Iberia.

ids_iberia = buses_iberia.index


# Seleccionamos las líneas cuyo bus inicial (bus0)
# Y cuyo bus final (bus1) pertenecen ambos a Iberia.
#
# El operador & significa "AND".
# Por tanto, esta condición significa:
#   bus0 está en España/Portugal
#   Y
#   bus1 está en España/Portugal
#
# El resultado son únicamente líneas internas de la Península Ibérica.

lineas_iberia = n_solved_2050.lines[
    n_solved_2050.lines.bus0.isin(ids_iberia) &
    n_solved_2050.lines.bus1.isin(ids_iberia)
]

# Mostramos las líneas internas seleccionadas.
lineas_iberia


# Ahora buscamos las líneas que cruzan el límite de Iberia.
#
# El operador ^ significa XOR ("exclusive OR").
# Esto significa que SOLO UNO de los dos extremos pertenece a Iberia:
#
#   bus0 está en Iberia y bus1 fuera
#       O
#   bus1 está en Iberia y bus0 fuera
#
# Estas son las posibles interconexiones internacionales.
#
# Usamos .copy() para crear una copia independiente del DataFrame,
# ya que después vamos a añadir columnas nuevas.

lineas_interconexion = n_solved_2050.lines[
    n_solved_2050.lines.bus0.isin(ids_iberia) ^
    n_solved_2050.lines.bus1.isin(ids_iberia)
].copy()


# Añadimos una columna llamada "country0".
# Para cada línea buscamos el país al que pertenece su bus0.
#
# linea.bus0 contiene el ID del bus.
# .map(n_solved_2050.buses.country) utiliza ese ID para buscar
# su país en la tabla de buses.

lineas_interconexion["country0"] = lineas_interconexion.bus0.map(
    n_solved_2050.buses.country
)


# Hacemos lo mismo para el segundo extremo de cada línea (bus1).

lineas_interconexion["country1"] = lineas_interconexion.bus1.map(
    n_solved_2050.buses.country
)


# Mostramos únicamente las columnas necesarias para comprobar
# qué países conecta cada línea candidata a interconexión.

lineas_interconexion[["bus0", "bus1", "country0", "country1"]]


# De todas las líneas que salen de Iberia,
# nos quedamos únicamente con aquellas en las que
# uno de los dos extremos pertenece a Francia (FR).
#
# El operador | significa "OR".
#
# Por tanto:
#   country0 es Francia
#       O
#   country1 es Francia

interconexiones_francia = lineas_interconexion[
    (lineas_interconexion.country0 == "FR") |
    (lineas_interconexion.country1 == "FR")
]


# Mostramos las interconexiones Iberia-Francia encontradas.
interconexiones_francia


# Finalmente imprimimos un pequeño resumen para comprobar
# cuántos elementos hemos seleccionado.

print("Buses España + Portugal:", len(buses_iberia))
print("Líneas internas Iberia:", len(lineas_iberia))
print("Interconexiones con Francia:", len(interconexiones_francia))

In [ ]:
# Creamos un conjunto con todos los buses que aparecen en las interconexiones con Francia.
#
# Tomamos:
#   - todos los bus0
#   - todos los bus1
#
# y los unimos en una sola lista.

buses_francia_conectados = set(
    interconexiones_francia.bus0.tolist()
    + interconexiones_francia.bus1.tolist()
)

# De ese conjunto eliminamos todos los buses que ya pertenecen a España o Portugal.
#
# El resultado son únicamente los buses franceses que están conectados directamente con la Península Ibérica.

buses_francia_conectados = buses_francia_conectados - set(ids_iberia)


# Seleccionamos en la tabla original de buses de PyPSA únicamente esos buses franceses de frontera.
#
# .loc[...] permite buscar filas por su identificador.
#
# Convertimos el set en lista porque .loc trabaja mejor con una colección ordenada de identificadores.

buses_francia = n_solved_2050.buses.loc[
    list(buses_francia_conectados)
]


# Unimos los buses de España y Portugal con los buses franceses necesarios para cerrar las interconexiones.
#
# pd.concat apila ambos DataFrames por filas.

buses_seleccionados = pd.concat([
    buses_iberia,
    buses_francia
])


# Hacemos lo mismo con las líneas:
#
#   - líneas internas de España y Portugal
#   - interconexiones entre Iberia y Francia
#
# El resultado contiene toda la red que queremos visualizar.

lineas_seleccionadas = pd.concat([
    lineas_iberia,
    interconexiones_francia
])


# Mostramos un pequeño resumen para comprobar que la selección final tiene sentido.

print("Buses totales mostrados:", len(buses_seleccionados))
print("Líneas totales mostradas:", len(lineas_seleccionadas))
print("Buses franceses frontera:", len(buses_francia))

## Plot

In [ ]:
import plotly.express as px

df_buses = buses_seleccionados.copy()
df_buses["bus"] = df_buses.index

fig = px.scatter_map(
    df_buses,
    lat="y",
    lon="x",
    hover_name="bus",
    hover_data={
        "country": True,
        "v_nom": True,
        "carrier": True,
        "x": False,
        "y": False,
    },
    zoom=5,
    height=800,
)

fig.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# -----------------------------
# PREPARAR BUSES
# -----------------------------

df_buses_iberia = buses_iberia.copy()
df_buses_iberia["bus"] = df_buses_iberia.index

df_buses_fr = buses_francia.copy()
df_buses_fr["bus"] = df_buses_fr.index

# -----------------------------
# PREPARAR LÍNEAS
# -----------------------------

line_lons = []
line_lats = []
line_text = []

for nombre_linea, linea in lineas_seleccionadas.iterrows():

    bus0 = n_solved_2050.buses.loc[linea.bus0]
    bus1 = n_solved_2050.buses.loc[linea.bus1]

    line_lons.extend([bus0.x, bus1.x, None])
    line_lats.extend([bus0.y, bus1.y, None])

    texto = (
        f"<b>Línea:</b> {nombre_linea}<br>"
        f"<b>Bus 0:</b> {linea.bus0}<br>"
        f"<b>Bus 1:</b> {linea.bus1}<br>"
    )

    if "length" in linea.index:
        texto += f"<b>Longitud:</b> {linea.length:.2f} km<br>"

    if "v_nom" in linea.index:
        texto += f"<b>v_nom:</b> {linea.v_nom} kV<br>"

    line_text.extend([texto, texto, None])


# -----------------------------
# CREAR FIGURA
# -----------------------------

fig = go.Figure()


# -----------------------------
# LÍNEAS
# -----------------------------

fig.add_trace(
    go.Scattermap(
        lon=line_lons,
        lat=line_lats,
        mode="lines",
        line=dict(width=1.2),
        name="Líneas",
        text=line_text,
        hoverinfo="text"
    )
)


# -----------------------------
# BUSES IBERIA
# -----------------------------

fig.add_trace(
    go.Scattermap(
        lon=df_buses_iberia["x"],
        lat=df_buses_iberia["y"],
        mode="markers",
        name="Buses España + Portugal",
        marker=dict(size=6),

        text=df_buses_iberia["bus"],

        customdata=df_buses_iberia[
            ["country", "carrier", "v_nom", "sub_network", "x", "y"]
        ].values,

        hovertemplate=
            "<b>Bus:</b> %{text}<br>" +
            "<b>País:</b> %{customdata[0]}<br>" +
            "<b>Carrier:</b> %{customdata[1]}<br>" +
            "<b>Tensión nominal:</b> %{customdata[2]} kV<br>" +
            "<b>Subnetwork:</b> %{customdata[3]}<br>" +
            "<b>Longitud:</b> %{customdata[4]:.4f}<br>" +
            "<b>Latitud:</b> %{customdata[5]:.4f}" +
            "<extra></extra>"
    )
)


# -----------------------------
# BUSES FRANCIA
# -----------------------------

fig.add_trace(
    go.Scattermap(
        lon=df_buses_fr["x"],
        lat=df_buses_fr["y"],
        mode="markers",
        name="Buses franceses de interconexión",
        marker=dict(size=10),

        text=df_buses_fr["bus"],

        customdata=df_buses_fr[
            ["country", "carrier", "v_nom", "sub_network", "x", "y"]
        ].values,

        hovertemplate=
            "<b>Bus:</b> %{text}<br>" +
            "<b>País:</b> %{customdata[0]}<br>" +
            "<b>Carrier:</b> %{customdata[1]}<br>" +
            "<b>Tensión nominal:</b> %{customdata[2]} kV<br>" +
            "<b>Subnetwork:</b> %{customdata[3]}<br>" +
            "<b>Longitud:</b> %{customdata[4]:.4f}<br>" +
            "<b>Latitud:</b> %{customdata[5]:.4f}" +
            "<extra></extra>"
    )
)


# -----------------------------
# CENTRO DEL MAPA
# -----------------------------

centro_lon = buses_seleccionados["x"].mean()
centro_lat = buses_seleccionados["y"].mean()


# -----------------------------
# CONFIGURACIÓN
# -----------------------------

fig.update_layout(

    title="PyPSA-Eur solved_2050.nc - Red ibérica e interconexiones con Francia",

    map=dict(
        style="open-street-map",
        center=dict(
            lat=centro_lat,
            lon=centro_lon
        ),
        zoom=5
    ),

    height=850,

    margin=dict(
        l=0,
        r=0,
        t=50,
        b=0
    ),

    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()

## Inspección de datos de la red solved_2050

In [ ]:
resumen = pd.DataFrame({
    "Componente": [
        "Buses",
        "Lines",
        "Links",
        "Transformers",
        "Carriers",
        "SubNetworks",
        "Shapes"
    ],
    "Cantidad": [
        len(n_solved_2050.buses),
        len(n_solved_2050.lines),
        len(n_solved_2050.links),
        len(n_solved_2050.transformers),
        len(n_solved_2050.carriers),
        len(n_solved_2050.sub_networks),
        len(n_solved_2050.shapes)
    ]
})

resumen

In [ ]:
buses_seleccionados.head(1)

In [ ]:
lineas_seleccionadas.head(1)

In [ ]:
n_solved_2050.transformers.head(1)

In [ ]:
n_solved_2050.links.head(1)

In [ ]:
n_solved_2050.carriers

In [ ]:
n_solved_2050.shapes